# DS207 Final Project - Predicting Readmission Rate for Diabetic Patients Using Machine Learning
#### Contributor: Rahil Sharma (rahilsharma@berkeley.edu)

### Notebook Structure:

1. Data Processing:
2. Baseline Model: 
    * Majority
    * Multiclass Logistic Regression
3. Notebook Exports:
    * Datasets: `results/train.csv`, `results/val.csv`, and `results/test.csv`
    * Baseline model: in `baseline.pkl`

## 1. Data Processing

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import joblib



In [ ]:
df = pd.read_csv('../data/diabetic_data.csv')
raw_df = pd.read_csv('../data/diabetic_data.csv')
display(df.head())

### Key for Drug descriptions ##

Up: The dosage of the drug was increased for the patient during their encounter.

Down: The dosage was decreased.

Steady: The patient is on the drug, and the dosage was not changed.

No: The patient was not prescribed this specific drug.

In [ ]:
print(f"There are {len(df.columns)} columns:\n", df.columns)

#### Drop duplicates and unnecessary columns

In [ ]:
df = df.drop_duplicates(subset=['patient_nbr'], keep='first')

In [ ]:
# checking to see how many missing values there are
(df == '?').sum()

# columns to keep based on descriptions / number of missing values
  # race (split into indicator variables)
  # gender (split into indicator variables)
  # age (split into indicator variables)
  # time_in_hospital (int)
  # num_lab_procedures (int)
  # num_procedures (int)
  # num_medications (int)
  # number_outpatient (int)
  # number_emergency (int)
  # number_inpatient (int)
  # number_diagnoses (int)
  # keeping all the indicator variables for medication / medication changes
  # target - multiclass classification

# dropping the unnecessary columns
df = df.drop(columns=['encounter_id', 'patient_nbr', 'weight', 'admission_type_id',
                      'discharge_disposition_id', 'admission_source_id', 'payer_code',
                      'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum',
                      'A1Cresult'],
             axis=1)

#### Collapse age and race

In [ ]:
# Convert age buckets like "[70-80)" to numeric midpoint (75)
def age_to_mid(age_bucket):
    if pd.isna(age_bucket):
        return np.nan
    try:
        lo, hi = age_bucket.strip('[]').split('-')
        lo = int(lo)
        hi = int(hi.strip(')'))
        return (lo + hi) / 2
    except Exception:
        return np.nan
df['age_mid'] = df['age'].apply(age_to_mid)
df.drop(columns=['age'], inplace=True)

In [ ]:
# Collapse low count race categories into 'Other'
if 'race' in df.columns:
    df['race'] = df['race'].fillna('Unknown')
    top = df['race'].value_counts().nlargest(4).index
    df['race_collapsed'] = df['race'].where(df['race'].isin(top), other='Other')
    df.drop(columns=['race'], inplace=True)

#### Categorical Features

In [ ]:
# creating indicator variables for the categorical features (one-hot encoding)
df = pd.get_dummies(df, columns=['race_collapsed', 'gender'], dtype=int)

# Encode ordinal values according to the prescription status of the visit
prescription_map = {'No': 0,       # The drug was not prescribed
                    'Down': 1,     # The dosage was decreased
                    'Steady': 2,   # The dosage did not change
                    'Up': 3}       # The dosage was increased during the encounter
cat_columns = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
               'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
               'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
               'miglitol', 'troglitazone', 'tolazamide', 'examide',
               'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
               'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
for cat_col in cat_columns:
    df[cat_col] = df[cat_col].map(prescription_map)

#### Binary features and Outcome of Interest

In [ ]:
# displaying the data
print("The number of columns are: ", len(df.columns))
display(df.head())
print(df.columns)

# recoding change and diabetesMed to 0 for No and 1 to Yes
df['change'] = np.where(df['change'] == 'No', 0, 1)
df['diabetesMed'] = np.where(df['diabetesMed'] == 'No', 0, 1)

# mapping the target into three classes
readmission_map = {
    'NO': 0,  # no readmission recorded
    '>30': 1, # readmitted (over 30 days)
    '<30': 1  # readmitted (within 30 days)
}

df['readmitted'] = df['readmitted'].map(readmission_map)

display(df.head())

df['readmitted'].value_counts()


In [ ]:
df.describe()

#### Shuffling and splitting

In [ ]:
# randomly shuffling the data
display(df.head())

indices = np.arange(len(df))

shuffled_indices = np.random.permutation(indices)

df = df.iloc[shuffled_indices].reset_index(drop=True)

display(df.head())

In [ ]:
# splitting X and Y data
X = df.copy().drop(columns=['readmitted'], axis=1)
Y = df.copy()['readmitted']

In [ ]:
# splitting the data into train, val, test (60/20/20)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.8, random_state=1234)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, train_size=0.75, random_state=1234)

#### Continuous feature standardizations

In [ ]:
# standardizing the continous features between 0 and 1
columns_to_standardize = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
                          'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age_mid']

scaler = MinMaxScaler()

X_train[columns_to_standardize] = scaler.fit_transform(X_train[columns_to_standardize])
X_val[columns_to_standardize] = scaler.transform(X_val[columns_to_standardize])
X_test[columns_to_standardize] = scaler.transform(X_test[columns_to_standardize])

print(f"The shape of X_train is {X_train.shape}")
print(f"\nThe shape of Y_train is {Y_train.shape}")
print(f"\nThe shape of X_val is {X_val.shape}")
print(f"\nThe shape of Y_val is {Y_val.shape}")
print(f"\nThe shape of X_test is {X_test.shape}")
print(f"\nThe shape of Y_test is {Y_test.shape}")


In [ ]:
display(X_train.head())
display(Y_train.head())

## 2. Baseline Model Developments

### 2.1 Majority

### 2.2 Multiclass logistic regression

In [ ]:
def build_model(learning_rate=0.01):

  tf.keras.backend.clear_session()

  model = tf.keras.Sequential()

  model.add(keras.Input(shape=(X_train.shape[1],)))

  model.add(keras.layers.Dense(
      units=64,
      activation='relu'
  ))

  model.add(keras.layers.BatchNormalization())
  model.add(keras.layers.Dropout(0.2))


  model.add(keras.layers.Dense(
      units=32,
      activation='relu'
  ))

  model.add(keras.layers.BatchNormalization())
  model.add(keras.layers.Dropout(0.2))


  model.add(keras.layers.Dense(
      units=16,
      activation='relu'
  ))

  # model.add(keras.layers.BatchNormalization())
  # model.add(keras.layers.Dropout(0.2))

  model.add(keras.layers.Dense(
      units=1,
      activation='sigmoid',
      kernel_initializer='glorot_uniform',
      bias_initializer='glorot_uniform'
  ))

  model.compile(
      optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
      loss=keras.losses.BinaryCrossentropy(),
      metrics=['accuracy']
  )

  history = model.fit(
      x=X_train,
      y=Y_train,
      validation_data=(X_val, Y_val),
      batch_size=64,
      epochs=10,
      verbose=1
  )

  return model, history

In [ ]:
m1, history = build_model(0.001)

In [ ]:
preds = (m1.predict(X_val) > 0.5).astype(int)
print(np.unique(preds, return_counts=True))

## 3. Notebook Exports

#### 3.1 Export the training, validation, and testing datasets

In [ ]:
df_train = pd.concat([X_train, Y_train], axis=1).to_csv('../results/train.csv', index=False)
df_val = pd.concat([X_val, Y_val], axis=1).to_csv('../results/val.csv', index=False)
df_test = pd.concat([X_test, Y_test], axis=1).to_csv('../results/test.csv', index=False)

#### 3.2 Export baseline model(s)

In [ ]:
joblib.dump(m1, '../results/baseline.pkl')